# Extração IBGE — População 10ª CRS (Arquitetura Medallion)

**Fonte:** API SIDRA do IBGE (`apisidra.ibge.gov.br`) — tabela 9514 (População residente, Censo).

**Período:** `p/last` — o notebook extrai automaticamente o **ano mais recente disponível** na tabela (hoje, 2022). O ano extraído é gravado na coluna `ano` e compõe o nome das tabelas bronze/silver.

## Fluxo

| Camada | Tabela | Conteúdo |
| --- | --- | --- |
| **Bronze** (Raw) | `bronze.dim_populacao_ibge_{ANO}_raw` | Dados brutos: ambos os sexos, todas as idades e formas de declaração, 11 municípios da 10ª CRS. Fiel e imutável em relação à fonte (únicas adições: chave técnica `cd_ibge6` e `dt_ingestao`). |
| **Silver** (Cleaned & Curated) | `silver.dim_populacao_ibge_{ANO}` | Filtros de negócio do rastreamento (sexo feminino, 25 a 64 anos), tipagem numérica, padronização (nome do município sem “ - RS”), grão município × faixa etária quinquenal. |
| **Gold** | `gold.dim_populacao_municipios` | Dimensão analítica por município: população feminina total e mulheres 25–64 (denominador das métricas de cobertura). |

> **Notas técnicas:** o endpoint `servicodados.ibge.gov.br` (API v3) não responde do compute do Databricks — usa-se `apisidra.ibge.gov.br` (API SIDRA v1), cujo código de município tem 7 dígitos (filtra-se pelos 6 primeiros, padrão SISCAN). Faixas quinquenais de 25–64 anos = categorias SIDRA **93088–93095** (93086–93093 seria 15–54 anos).

## Resumo — o que este notebook faz

1. Consulta a **API do IBGE** (SIDRA, tabela 9514 — Censo) e detecta automaticamente o **ano mais recente** com dados (`p/last`).
2. **Bronze** — salva os dados brutos: população por sexo e idade (todas as faixas) dos 11 municípios da 10ª CRS, fiel à fonte.
3. **Silver** — aplica os filtros de negócio do rastreamento do colo do útero: **mulheres de 25 a 64 anos**, por município e faixa etária, com tipagem e padronização.
4. **Gold** — monta a dimensão `gold.dim_populacao_municipios` (população feminina e mulheres 25–64 por município), usada como denominador das métricas de cobertura dos exames.

Basta executar as células em ordem, de cima para baixo.

## Setup — municípios e funções de acesso à API

Esta célula prepara tudo o que as demais usam:

* **`MUNICIPIOS_10CRS`** — dicionário com os códigos IBGE (6 dígitos) dos 11 municípios da 10ª CRS.
* **`BASE_SIDRA`** — endereço da API SIDRA v1 do IBGE (o endpoint v3 não responde do compute do Databricks).
* **`sidra_get(path)`** — faz a requisição HTTP com até 3 tentativas automáticas em caso de instabilidade da API.
* **`parse_sidra(data)`** — converte o JSON retornado pelo IBGE em DataFrame com colunas rotuladas.
* **`filtrar_10crs(df)`** — mantém apenas os 11 municípios da 10ª CRS (a API usa códigos de 7 dígitos; o filtro pega os 6 primeiros, padrão SISCAN).

In [0]:
import requests
import time
import pandas as pd

# 1. Códigos IBGE (6 dígitos) dos 11 municípios da 10ª CRS (RS)
MUNICIPIOS_10CRS = {
    "430040": "Alegrete",
    "430187": "Barra do Quaraí",
    "431060": "Itaqui",
    "431171": "Maçambara",
    "431175": "Manoel Viana",
    "431530": "Quaraí",
    "431640": "Rosário do Sul",
    "431697": "Santa Margarida do Sul",
    "431710": "Sant'Ana do Livramento",
    "431830": "São Gabriel",
    "432240": "Uruguaiana"
}

# Endpoint da API SIDRA v1 — o servicodados.ibge.gov.br (v3) fica inacessível
# a partir do compute do Databricks (timeout de conexão)
BASE_SIDRA = "https://apisidra.ibge.gov.br/values"


def sidra_get(path, tentativas=3, timeout=120):
    """Consulta a API SIDRA v1 e retorna o JSON (a linha 0 é o cabeçalho).

    A API do IBGE sofre instabilidades transitórias: erros de conexão/timeout
    são retentados automaticamente com backoff. Erros HTTP (ex.: 400) são
    determinísticos e propagam imediatamente, sem retry.
    """
    ultimo_erro = None
    for i in range(tentativas):
        try:
            r = requests.get(f"{BASE_SIDRA}{path}", timeout=timeout)
            r.raise_for_status()
            return r.json()
        except (requests.ConnectionError, requests.Timeout) as e:
            ultimo_erro = e
            print(f"sidra_get: tentativa {i + 1}/{tentativas} falhou ({type(e).__name__}); repetindo...")
            time.sleep(3 * (i + 1))
    raise ultimo_erro


def parse_sidra(data):
    """Converte a resposta da SIDRA v1 em DataFrame com colunas rotuladas."""
    headers = data[0]  # {"V": "Valor", "D1C": "Município (Código)", ...}
    return pd.DataFrame([{headers[k]: v for k, v in row.items()} for row in data[1:]])


def filtrar_10crs(df):
    """Mantém apenas os municípios da 10ª CRS.

    Na API SIDRA o código de município tem 7 dígitos (com dígito verificador);
    os 6 primeiros dígitos são o código IBGE padrão usado nas tabelas SISCAN.
    """
    df = df.copy()
    df["cd_ibge6"] = df["Município (Código)"].str[:6]
    return df[df["cd_ibge6"].isin(MUNICIPIOS_10CRS)]


## Bronze — dados brutos (Raw)

Extração **completa e sem filtros de negócio**: ambos os sexos (c2/4,5), todas as categorias de idade (c287/all — “Total”, faixas quinquenais, idades simples) e todas as formas de declaração, para os 11 municípios da 10ª CRS. O dado permanece **fiel e imutável** em relação à fonte oficial; as únicas adições são a chave técnica `cd_ibge6` e a marcação `dt_ingestao`.

In [0]:
# ==============================================================================
# BRONZE — extração bruta: ambos os sexos, TODAS as idades, 11 municípios
# ==============================================================================
# Sem nenhum filtro de negócio: c287/all traz todas as categorias de idade
# (Total, quinquenais, idades simples) e ambos os sexos.
# A API SIDRA às vezes recusa a combinação c2/4,5 + c287/all numa única
# chamada; nesse caso, faz uma consulta por sexo e concatena.
try:
    df_bronze_raw = filtrar_10crs(parse_sidra(sidra_get("/t/9514/n6/in n3 43/v/93/p/last/c2/4,5/c287/all")))
    print("Bronze: consulta única OK")
except requests.HTTPError as e:
    print("Consulta única recusada pela API:", e.response.status_code, "-", e.response.text[:150])
    print("Usando fallback: uma consulta por sexo...")
    partes = [
        filtrar_10crs(parse_sidra(sidra_get(f"/t/9514/n6/in n3 43/v/93/p/last/c2/{sexo}/c287/all")))
        for sexo in ("4", "5")
    ]
    df_bronze_raw = pd.concat(partes, ignore_index=True)
    print("Bronze: fallback por sexo OK")

# Ano resolvido automaticamente pela API (p/last)
ANO = df_bronze_raw["Ano"].iloc[0]
print(f"Ano mais recente com dados na tabela 9514: {ANO}")
print("Linhas brutas (10ª CRS):", len(df_bronze_raw))
print("Sexos:", sorted(df_bronze_raw["Sexo"].unique()))
print("Categorias de idade distintas:", df_bronze_raw["Idade"].nunique())
print("Formas de declaração da idade:", sorted(df_bronze_raw["Forma de declaração da idade"].unique()))

# Únicas adaptações à fonte: chave cd_ibge6 (6 dígitos, padrão SISCAN),
# dt_ingestao e nomes de coluna em snake_case (o Delta não aceita espaços,
# parênteses e acentos). Os VALORES permanecem exatamente como retornados.
import re
import unicodedata

def slug(nome):
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    return re.sub(r"[^0-9a-zA-Z]+", "_", nome).strip("_").lower()

df_bronze = df_bronze_raw.copy()
df_bronze["dt_ingestao"] = pd.Timestamp.now(tz="America/Sao_Paulo")
df_bronze.columns = [slug(c) for c in df_bronze.columns]
print("Colunas da bronze:", list(df_bronze.columns))

bronze_table = f"`tcc-analise-siscan-10crs`.bronze.dim_populacao_ibge_{ANO}_raw"
spark.createDataFrame(df_bronze).write.format("delta").mode("overwrite").saveAsTable(bronze_table)
print("Bronze salva em:", bronze_table)

display(df_bronze.head(20))

## Silver — dados tratados (Cleaned & Curated)

Partindo da **Bronze**, aplicam-se os filtros de negócio do rastreamento do câncer do colo do útero: **sexo feminino**, faixa etária **25 a 64 anos** (categorias SIDRA 93088–93095) e forma de declaração “Total” (evita dupla contagem entre esquemas de idade). Depois: tipagem numérica, padronização do nome do município (sem sufixo “ - RS”) e colunas técnicas. Grão: **município × faixa etária quinquenal**.

In [0]:
# ==============================================================================
# SILVER — leitura da Bronze + filtros de negócio, tipagem e padronização
# ==============================================================================
# Filtro do rastreamento do colo do útero: sexo feminino, 25 a 64 anos.
FAIXAS_25_64 = ["93088", "93089", "93090", "93091", "93092", "93093", "93094", "93095"]  # 25-29 ... 60-64

df_bronze_in = spark.table(bronze_table).toPandas()

df_silver = df_bronze_in[
    (df_bronze_in["sexo"] == "Mulheres")
    & (df_bronze_in["forma_de_declaracao_da_idade"] == "Total")
    & (df_bronze_in["idade_codigo"].isin(FAIXAS_25_64))
].copy()

# Fail-fast: se um futuro período do agregado 9514 usar códigos de categoria
# de idade diferentes, este filtro retornaria 0 linhas silenciosamente.
assert len(df_silver) == 88, (
    f"Esperado 88 linhas (11 municípios × 8 faixas 25-64); obtido: {len(df_silver)}. "
    f"Verificar se as categorias de idade do período {ANO} mudaram na API SIDRA."
)

# Tipagem e padronização
df_silver["populacao_mulheres"] = pd.to_numeric(df_silver["valor"], errors="raise").astype("int64")
df_silver = df_silver.assign(
    cd_ibge=df_silver["cd_ibge6"],
    municipio=df_silver["municipio"].str.replace(" - RS", "", regex=False).str.strip(),
    cd_faixa_ibge=df_silver["idade_codigo"],
    faixa_etaria=df_silver["idade"],
    faixa_inicio=df_silver["idade"].str.extract(r"^(\d+) a \d+ anos$")[0].astype("int64"),
    faixa_fim=df_silver["idade"].str.extract(r"^\d+ a (\d+) anos$")[0].astype("int64"),
    ano=df_silver["ano"].astype("int64"),
)[["cd_ibge", "municipio", "cd_faixa_ibge", "faixa_etaria", "faixa_inicio", "faixa_fim", "populacao_mulheres", "ano", "dt_ingestao"]]

print("Linhas silver:", len(df_silver), "(esperado: 11 municípios × 8 faixas = 88)")
print("Faixas etárias:", sorted(df_silver["faixa_etaria"].unique()))

silver_table = f"`tcc-analise-siscan-10crs`.silver.dim_populacao_ibge_{ANO}"
spark.createDataFrame(df_silver).write.format("delta").mode("overwrite").saveAsTable(silver_table)
print("Silver salva em:", silver_table)

display(df_silver.sort_values(["municipio", "faixa_inicio"]))

## Gold — dimensão analítica

Agrega a **Silver** (mulheres 25–64 por município) e a **Bronze** (população feminina total — linha `Idade = "Total"`) na dimensão `gold.dim_populacao_municipios`, denominador das métricas de cobertura do rastreamento (exames por mulher em idade-alvo).

In [0]:
# ==============================================================================
# GOLD — dimensão analítica por município (construída da Silver + Bronze)
# ==============================================================================
# Mulheres 25-64: agregação da Silver.
# População feminina total: linha Idade="Total" (sexo feminino) da Bronze.
df_silver_in = spark.table(silver_table).toPandas()

df_mulheres_25_64 = (
    df_silver_in.groupby(["cd_ibge", "municipio", "ano"], as_index=False)["populacao_mulheres"]
    .sum()
    .rename(columns={"populacao_mulheres": "mulheres_25_64_anos"})
)

df_fem_total = (
    df_bronze_in[(df_bronze_in["sexo"] == "Mulheres") & (df_bronze_in["idade"] == "Total")]
    .assign(populacao_feminina=lambda d: d["valor"].astype("int64"))
    .rename(columns={"cd_ibge6": "cd_ibge"})
    [["cd_ibge", "populacao_feminina"]]
)

df_populacao_municipios = df_fem_total.merge(
    df_mulheres_25_64[["cd_ibge", "municipio", "ano", "mulheres_25_64_anos"]], on="cd_ibge"
)
df_populacao_municipios = df_populacao_municipios[["cd_ibge", "municipio", "populacao_feminina", "ano", "mulheres_25_64_anos"]]
df_populacao_municipios = df_populacao_municipios.sort_values("populacao_feminina", ascending=False).reset_index(drop=True)

gold_table = "`tcc-analise-siscan-10crs`.gold.dim_populacao_municipios"
# overwriteSchema: a coluna `ano` passa de string para int (tipagem padronizada na Silver)
spark.createDataFrame(df_populacao_municipios).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table)
print("Gold salva em:", gold_table)

display(df_populacao_municipios)